In [6]:
! pip install finnhub-python pandas numpy matplotlib ipywidgets

! pip install "numpy<2" --upgrade --force-reinstall


  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
gradio 5.20.1 requires typer<1.0,>=0.12; sys_platform != "emscripten", but you have typer 0.9.0 which is incompatible.


In [5]:
import finnhub
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
from datetime import datetime


In [ ]:
# Connecting to Finnhub API
API_KEY = "d3mum09r01qmso35hdugd3mum09r01qmso35hdv0"
finnhub_client = finnhub.Client(api_key=API_KEY)

# Testing connection by fetching Apple stock
quote = finnhub_client.quote("AAPL")
print("Finnhub Connected Successfully!")
print("Live Quote Data:\n", quote)


Finnhub Connected Successfully!
Live Quote Data:
 {'c': 247.66, 'd': 2.39, 'dp': 0.9744, 'h': 249.69, 'l': 245.56, 'o': 249.38, 'pc': 245.27, 't': 1760385600}


In [8]:
# Fetching multiple stocks and calculate ROI
def get_stock_data(symbols=["AAPL", "AMZN", "TSLA", "MSFT", "GOOG"]):
    """Fetching live stock data for multiple symbols from Finnhub."""
    rows = []
    for sym in symbols:
        q = finnhub_client.quote(sym)
        rows.append({
            "Stock": sym,
            "Price": q["c"],
            "PrevClose": q["pc"],
        })

    df = pd.DataFrame(rows)
    df["ROI"] = (df["Price"] - df["PrevClose"]) / df["PrevClose"] * 100
    df["Satisfaction"] = np.random.uniform(3, 5, len(df))
    return df

# Run and test
df = get_stock_data()
display(df)


,Stock,Price,PrevClose,ROI,Satisfaction
0,AAPL,247.66,245.27,0.974436,4.161077
1,AMZN,220.07,216.37,1.710034,4.380447
2,TSLA,435.90,413.49,5.419720,4.308705
3,MSFT,514.05,510.96,0.604744,3.689756
4,GOOG,244.64,237.49,3.010653,3.435213


In [12]:
# Step 4: Normalize and calculate AHP scores

def recalc_AHP(df, weights):
    """Recalculating AHP scores based on ROI, cost, and satisfaction."""
    df["ROI_norm"] = df["ROI"] / df["ROI"].max()
    df["Cost_norm"] = 1 - (df["Price"] / df["Price"].max())
    df["Satisfaction_norm"] = df["Satisfaction"] / df["Satisfaction"].max()

    # Final AHP Score
    df["AHP_Score"] = (weights[0]*df["ROI_norm"] +
                       weights[1]*df["Cost_norm"] +
                       weights[2]*df["Satisfaction_norm"])
    return df

weights = np.array([0.4, 0.3, 0.3])

df = recalc_AHP(df, weights)
display(df[["Stock", "ROI_norm", "Cost_norm", "Satisfaction_norm", "AHP_Score"]])


,Stock,ROI_norm,Cost_norm,Satisfaction_norm,AHP_Score
0,AAPL,0.179795,0.518218,0.949921,0.512359
1,AMZN,0.315521,0.571890,1.000000,0.597775
2,TSLA,1.000000,0.152028,0.983622,0.740695
3,MSFT,0.111582,0.000000,0.842324,0.297330
4,GOOG,0.555500,0.524093,0.784215,0.614692


In [13]:
# Ranking the stocks based on AHP Score (descending order)
ranked_df = df.sort_values(by="AHP_Score", ascending=False).reset_index(drop=True)

print("Top Stocks Ranked by AHP Score:\n")
display(ranked_df[["Stock", "Price", "ROI", "Satisfaction", "AHP_Score"]])


Top Stocks Ranked by AHP Score:



,Stock,Price,ROI,Satisfaction,AHP_Score
0,TSLA,435.90,5.419720,4.308705,0.740695
1,GOOG,244.64,3.010653,3.435213,0.614692
2,AMZN,220.07,1.710034,4.380447,0.597775
3,AAPL,247.66,0.974436,4.161077,0.512359
4,MSFT,514.05,0.604744,3.689756,0.297330


In [18]:
# Initializing
weights = np.array([0.4, 0.3, 0.3])
learning_rate = 0.05 # this is an learning rate
threshold = 0.005

def update_weights(weights, reward):
    """Adjusting weights slightly based on reward feedback."""
    delta = learning_rate * reward
    new_weights = weights + delta * np.random.uniform(-1, 1, size=3)
    new_weights = np.clip(new_weights, 0.05, 0.8)
    new_weights /= new_weights.sum()
    return new_weights

print("Starting adaptive loop...\n")

for step in range(10):
    # Gets fresh data
    df_live = get_stock_data()
    df_live = recalc_AHP(df_live, weights)
    avg_before = df_live["AHP_Score"].mean()

    # Simulating small market drift
    time.sleep(30)
    df_next = get_stock_data()
    df_next = recalc_AHP(df_next, weights)
    avg_after = df_next["AHP_Score"].mean()

    # Reward = improvement in avg AHP score
    reward = avg_after - avg_before

    # Triggering weight update only if change is meaningful
    if abs(reward) > threshold:
        weights = update_weights(weights, reward)
        print(f"Step {step}: Updated (={reward:.5f}) | New weights = {weights}")
    else:
        print(f"Step {step}: Skipped update ({reward:.5f})")

print("\nFinal adaptive weights:", weights)


Starting adaptive loop...

Step 0: Updated (=-0.01407) | New weights = [0.39984859 0.30044592 0.29970549]
Step 1: Skipped update (0.00453)
Step 2: Updated (=-0.01616) | New weights = [0.40018086 0.30031856 0.29950058]
Step 3: Updated (=-0.00526) | New weights = [0.4000737  0.30037305 0.29955324]
Step 4: Skipped update (0.00044)
Step 5: Updated (=0.02784) | New weights = [0.3999652  0.30072603 0.29930876]
Step 6: Updated (=0.02091) | New weights = [0.40035756 0.30074972 0.29889271]
Step 7: Updated (=-0.02888) | New weights = [0.40061207 0.29959131 0.29979663]
Step 8: Skipped update (0.00234)
Step 9: Updated (=0.02296) | New weights = [0.39944987 0.30023296 0.30031718]

Final adaptive weights: [0.39944987 0.30023296 0.30031718]


In [20]:
def get_stock_data():
    symbols = ["AAPL", "AMZN", "TSLA", "MSFT", "GOOG"]
    rows = []
    for s in symbols:
        q = finnhub_client.quote(s)
        rows.append({
            "Stock": s,
            "Price": q["c"],
            "PrevClose": q["pc"],
            "ROI": ((q["c"] - q["pc"]) / q["pc"]) * 100,  # percentage return
            "Satisfaction": np.random.uniform(3.5, 5.0)   # mock satisfaction rating
        })
    df = pd.DataFrame(rows)
    return df


In [22]:
import time

learning_rate = 0.05
threshold = 0.002  # more sensitive trigger
weights = np.array([0.4, 0.3, 0.3])  # ROI, Cost, Satisfaction

for step in range(10):
    try:
        df_live = get_stock_data()
        df_live = recalc_AHP(df_live, weights)
        avg_before = df_live["AHP_Score"].mean()

        time.sleep(30)  # wait 30 sec before next fetch

        df_new = get_stock_data()
        df_new = recalc_AHP(df_new, weights)
        avg_after = df_new["AHP_Score"].mean()

        reward = avg_after - avg_before
        if abs(reward) > threshold:
            weights = update_weights(weights, reward)
            print(f"Step {step}: Update triggered ({reward:.4f}) | New Weights={weights}")
        else:
            print(f"Step {step}: Skipped update ({reward:.4f})")

    except Exception as e:
        print("Error fetching live data:", e)

print("Final Adaptive Weights:", weights)


Step 0: Update triggered (0.0193) | New Weights=[0.40063772 0.30004999 0.29931229]
Step 1: Update triggered (0.0133) | New Weights=[0.40082555 0.30018463 0.29898983]
Step 2: Update triggered (-0.0293) | New Weights=[0.40002101 0.30041308 0.29956591]
Step 3: Update triggered (-0.0107) | New Weights=[0.399881   0.30076063 0.29935837]
Step 4: Skipped update (0.0011)
Step 5: Update triggered (-0.0064) | New Weights=[0.40015443 0.30076481 0.29908076]
Step 6: Update triggered (0.0028) | New Weights=[0.40019364 0.30072812 0.29907825]
Step 7: Update triggered (0.0191) | New Weights=[0.4008685  0.30019649 0.298935  ]
Step 8: Update triggered (-0.0090) | New Weights=[0.40106909 0.30008759 0.29884332]
Step 9: Update triggered (-0.0132) | New Weights=[0.40097282 0.29985439 0.29917279]
Final Adaptive Weights: [0.40097282 0.29985439 0.29917279]


In [23]:
df_live.sort_values("AHP_Score", ascending=False).head(5)


,Stock,Price,PrevClose,ROI,Satisfaction,ROI_norm,Cost_norm,Satisfaction_norm,AHP_Score
2,TSLA,435.90,413.49,5.419720,4.042887,1.000000,0.152028,0.915058,0.720150
4,GOOG,244.64,237.49,3.010653,3.963428,0.555500,0.524093,0.897073,0.648152
1,AMZN,220.07,216.37,1.710034,3.621485,0.315521,0.571890,0.819679,0.543118
0,AAPL,247.66,245.27,0.974436,3.756976,0.179795,0.518218,0.850345,0.481741
3,MSFT,514.05,510.96,0.604744,4.418176,0.111582,0.000000,1.000000,0.343595
